# Step 6 overfitting study on Kaggle (GPU)

Settings (right panel): **Accelerator = GPU T4 x2** (or P100), **Internet = On** (needed to download the ResNet-50 weights).

Input: a Kaggle dataset containing `crops.zip` (or the unzipped `crops/` folder) and `train_classifier.py`.

Run it with **Save Version → Save & Run All (Commit)**: it runs in the background, and the results appear under the **Output** tab when it finishes.

In [ ]:
# 1. Check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Find the uploaded files, unzip crops if needed, copy the script
import glob, os, shutil, zipfile

script = glob.glob('/kaggle/input/**/train_classifier.py', recursive=True)
assert script, 'train_classifier.py not found in /kaggle/input - add your dataset with Add Input'
shutil.copy(script[0], '/kaggle/working/train_classifier.py')

# Kaggle usually unzips uploaded zip files itself; handle both cases
train_dirs = [d for d in glob.glob('/kaggle/input/**/train', recursive=True) if os.path.isdir(d)
              and os.path.isdir(d.replace('/train', '/val')) and 'paragraph' in os.listdir(d)]
if train_dirs:
    CROPS = os.path.dirname(train_dirs[0])
else:
    zips = glob.glob('/kaggle/input/**/crops.zip', recursive=True)
    assert zips, 'neither a crops folder nor crops.zip found in /kaggle/input'
    zipfile.ZipFile(zips[0]).extractall('/tmp')   # /tmp is not saved as output (keeps Output small)
    CROPS = '/tmp/crops'

os.chdir('/kaggle/working')
print('crops folder:', CROPS, '->', sorted(os.listdir(CROPS)))
print('classes:', len(os.listdir(os.path.join(CROPS, 'train'))))

In [ ]:
# Same settings as the Colab runs (C2 = layer4 setup); Kaggle has 4 CPU cores, so 4 workers
BASE = f'python train_classifier.py --model resnet50 --strategy layer4 --data {CROPS} --epochs 15 --patience 3 --workers 4'

In [ ]:
# G3b: L1 (lasso), stronger; 1e-5 adds about 3 to the loss
!{BASE} --name G3b_l1_1e-5 --l1 1e-5

In [ ]:
# G10: data augmentation (rotation, brightness/contrast, blur; no flips)
!{BASE} --name G10_augment --augment

In [ ]:
# Imbalance check: class weights OFF (compare with C2, which has them ON)
!{BASE} --name C2_noweights --no-class-weights

In [ ]:
# G4: elastic net = L1 + L2 together
!{BASE} --name G4_elastic --l1 1e-6 --l2 1e-2

In [ ]:
# G7: remove max-pool after the first conv layer (about 4x more activations: slowest run, so last)
!{BASE} --name G7_nopool --no-first-pool

In [ ]:
# Results of the runs done here, then one zip to download (Output tab)
import pandas as pd
print(pd.read_csv('runs_cls/summary.csv')[['name', 'best_epoch', 'train_acc', 'val_acc', 'gap', 'test_acc', 'test_macro_f1', 'sec_per_epoch']].to_string())
shutil.make_archive('/kaggle/working/runs_cls_kaggle', 'zip', '/kaggle/working', 'runs_cls')